In [1]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import Input
import os


2025-09-12 14:58:32.787802: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-12 14:58:32.797001: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757681912.809188  584790 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757681912.815514  584790 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-09-12 14:58:32.828430: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [2]:

# ----------------------------
# Config
# ----------------------------
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 50
LR = 1e-3
DROPOUT_RATE = 0.8
N_CLASSES = 3   # Normal, Benign, Malignant


In [3]:

# ----------------------------
# Data pipeline
# ----------------------------
train_datagen = ImageDataGenerator(
    rescale=1./255,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    "CDD-CESM/organized_images",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    color_mode="rgb"  # important : force 3 canaux
)

val_generator = train_datagen.flow_from_directory(
    "CDD-CESM/organized_images",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    color_mode="rgb"
)

batch_x, batch_y = next(train_generator)
print("Sample batch shape:", batch_x.shape)  # doit être (16,224,224,3)


Found 1601 images belonging to 3 classes.
Found 400 images belonging to 3 classes.
Sample batch shape: (16, 224, 224, 3)


In [4]:

# ----------------------------
# Model definition (patch skip_mismatch)
# ----------------------------
inputs = Input(shape=IMG_SIZE + (3,))
base_model = EfficientNetB0(weights=None, include_top=False, input_tensor=inputs)

# Télécharger les poids officiels
weights_path = tf.keras.utils.get_file(
    "efficientnetb0_notop.h5",
    "https://storage.googleapis.com/keras-applications/efficientnetb0_notop.h5"
)

# Charger les poids en ignorant les couches incompatibles (stem_conv)
base_model.load_weights(weights_path, by_name=True, skip_mismatch=True)
print("EfficientNetB0 loaded with skip_mismatch=True")

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(DROPOUT_RATE)(x)  # large dropout (0.8) as per paper
output = Dense(N_CLASSES, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

# Fine-tune ALL layers
for layer in base_model.layers:
    layer.trainable = True


2025-09-12 14:58:35.152652: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


EfficientNetB0 loaded with skip_mismatch=True


In [5]:

# ----------------------------
# Compile
# ----------------------------
optimizer = tf.keras.optimizers.Adam(learning_rate=LR)
model.compile(
    optimizer=optimizer,
    loss="categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.Precision(name="precision"),
        tf.keras.metrics.Recall(name="recall")
    ]
)


In [ ]:

# ----------------------------
# Train
# ----------------------------
callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.1, patience=5, verbose=1),
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)


/home/light/miniforge3/envs/pytorch/lib/python3.10/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50


/home/light/miniforge3/envs/pytorch/lib/python3.10/site-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor']
Received: inputs=Tensor(shape=(None, 224, 224, 3))
  warnings.warn(msg)


KeyboardInterrupt: 

: 

In [ ]:

# ----------------------------
# Evaluate
# ----------------------------
loss, acc, prec, rec = model.evaluate(val_generator)
print(f"Validation Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}, Recall: {rec:.4f}")


In [ ]:
from sklearn.metrics import confusion_matrix

# Get true labels and predictions from the validation generator
import numpy as np

val_steps = val_generator.samples // val_generator.batch_size
y_true = []
y_pred = []

for i in range(val_steps):
	x_batch, y_batch = next(val_generator)
	y_true.extend(np.argmax(y_batch, axis=1))
	preds = model.predict(x_batch)
	y_pred.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true, y_pred)
print(cm)